# Revision 2026-10-02 — Check IV: small-$n$ comparison RFM × DiffMeans (F3)

Implements the pre-registered design `IV-small-n_design.md` (decisions of 2026-10-08). Section numbers (§) below refer to that document; it is the source of truth for definitions and criteria.

**Question.** For balanced training subsamples with $n$ examples per class, does the RFM direction give a better test AUROC than DiffMeans?
$$\Delta_c(n)=\mathbb{E}_S\big[m(v_{\text{RFM}}(S))-m(v_{\text{DM}}(S))\big],\qquad m=\text{test AUROC (primary)},\ |\rho|\text{ (secondary)}.$$

**Model / data.** GuppyLM cached last-token activations, $d=384$. Primary block $\ell=3$; secondary: all 6 blocks, reported without selection. Concepts `valence`, `env_social` (main), `food` (ceiling control). No new inference.

**Criteria (fixed before running, §3.6).**
- **C0 (sanity, stops the run):** $n=1$: $1-|\cos(v_{\text{RFM}},v_{\text{DM}})|<10^{-8}$ and $|\Delta|<10^{-8}$ for every subsample. Every $n$: $\lVert P_{\mathcal D^\perp}v\rVert<10^{-6}$ for the four methods. Step 0: recomputed $(L,T)$ equal to the cache wherever `hp_normalize = 0`.
- **H3a** refuted if, for some main concept and $n\in\{2,4\}$, the lower 95% CI bound of $\Delta_c(n)$ is $>0$.
- **H-artigo** supported if Holm ($\alpha=0.025$, one-sided bootstrap $p$, $k=10$ cells) declares at least one cell with $\Delta_c(n)>0$; refuted otherwise.
- **H3b** supported if, in the same concept, one cell has upper CI bound $<0$ and another with larger $n$ has lower bound $>0$.
- **H3a, reinforcement (secondary):** for $n\in\{2,4\}$, the upper CI bound of $\hat\Delta^{\max}_c(n)$ (oracle $T$) is $<0$.

**How to run.** From `rfm_checks/`. Cost: ≈45 min with all 6 blocks, ≈7 min with `RUN_SECONDARY = False` (laptop CPU). The run stops with an `AssertionError` if C0 fails. `IV-small-n.md` is written by hand after the run.

In [ ]:
import sys, os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
try: import rfm_guppy.paths
except ModuleNotFoundError:   # repo root not on sys.path: walk up from cwd to the dir holding rfm_guppy/paths.py
    _r = os.getcwd()
    while not os.path.isfile(os.path.join(_r, "rfm_guppy", "paths.py")):
        if _r == os.path.dirname(_r): raise RuntimeError("Run this notebook from inside the repository.")
        _r = os.path.dirname(_r)
    sys.path.insert(0, _r)
from rfm_guppy.paths import ROOT, CACHE_DIR
from rfm_guppy.rfm import _agop_step, _preprocess, orient, extract_concept_vector
from rfm_guppy.baselines import logreg_vector
CACHE = CACHE_DIR
PREFIX = 'IV-small-n'

CONCEPTS = ['valence', 'env_social', 'food']; MAIN = ['valence', 'env_social']
NB, B_PRIMARY = 6, 3
RUN_SECONDARY = True                       # False: primary block only
BLOCKS = [B_PRIMARY] + ([b for b in range(NB) if b != B_PRIMARY] if RUN_SECONDARY else [])
N_GRID = [1, 2, 4, 8, 16, 32]; N_TEST = [2, 4, 8, 16, 32]
R, NBOOT, TMAX, LAM, ALPHA, SEED = 200, 10_000, 10, 1e-3, 0.025, 20261008
C_GRID = [1000.0, 100.0, 1.0, 0.1]
TOL_C0, TOL_RES = 1e-8, 1e-6
METHODS = ['RFM', 'DM', 'PCA', 'LogReg']

def load_split(c, s):
    ds = json.load(open(f'{CACHE}/concept_{c}.json'))[s]
    y = np.array([d['label'] for d in ds], int)
    cat = np.array([d['category'] for d in ds])
    Z = np.load(f'{CACHE}/{c}_{s}_activations.npy').astype(np.float64)
    assert Z.shape[0] == len(y)
    return Z, y, cat

TR = {c: load_split(c, 'train') for c in CONCEPTS}
VA = {c: load_split(c, 'val') for c in CONCEPTS}
VEC = {c: dict(np.load(f'{CACHE}/{c}_vectors.npz')) for c in CONCEPTS}
for c in CONCEPTS:
    print(f"{c:10s} train {TR[c][0].shape} pos {TR[c][1].sum()} | val {VA[c][0].shape} pos {VA[c][1].sum()}")
print('blocks:', BLOCKS)

## 0. Step 0: frozen hyperparameters (train + val only; test split still closed)

**RFM (§3.3).** `extract_concept_vector(..., sweep={"normalize": [False]})` on the full `train` split, selection by val $|\rho|$, $L\in\{1,10,100\}$, $T\in\{1..10\}$, $\lambda=10^{-3}$; ties go to the first combination in scan order (the code uses strict `>`). Where the cache has `hp_normalize = 0`, the recomputed $(L,T)$ must equal the cached values (part of C0).

**LogReg.** $C$ is not stored in the cache and `baselines.logreg_vector` does not return it, so the loop is reproduced here (same grid, solver and tie rule) and checked against `logreg_vector`.

The table is saved to `IV-small-n_hp.csv`.

In [ ]:
def logreg_select_C(Ztr, ytr, Zva, yva):
    Zc, mu = _preprocess(Ztr, True, False); Zvc, _ = _preprocess(Zva, True, False, mean_vec=mu)
    best_rho, best_C, best_v = -1.0, None, None
    for C in C_GRID:
        w = LogisticRegression(C=C, solver='liblinear', max_iter=2000, random_state=42).fit(Zc, ytr).coef_[0]
        if np.linalg.norm(w) < 1e-10: continue
        v = orient(w / np.linalg.norm(w), Zc, ytr); p = Zvc @ v
        rho = 0.0 if p.std() < 1e-12 else abs(float(np.corrcoef(p, yva)[0, 1]))
        if rho > best_rho: best_rho, best_C, best_v = rho, C, v
    return best_rho, best_C, best_v

HP, rows = {}, []; t0 = time.time()
for c in CONCEPTS:
    (Ztr, ytr, _), (Zva, yva, _), V = TR[c], VA[c], VEC[c]
    for b in range(NB):
        v, info = extract_concept_vector(Ztr[:, b], ytr, Zva[:, b], yva, sweep={'normalize': [False]})
        rho_lr, C, v_lr = logreg_select_C(Ztr[:, b], ytr, Zva[:, b], yva)
        v_ref, _, rho_ref = logreg_vector(Ztr[:, b], ytr, Zva[:, b], yva)
        assert np.allclose(v_lr, v_ref) and abs(rho_lr - rho_ref) < 1e-12, (c, b, 'LogReg loop differs from logreg_vector')
        cL, cT, cN = float(V['hp_L'][b]), int(V['hp_T'][b]), int(V['hp_normalize'][b])
        same = (float(info['L']) == cL) and (int(info['T']) == cT)
        HP[(c, b)] = dict(L=float(info['L']), T=int(info['T']), C=float(C))
        rows.append(dict(concept=c, block=b, L=float(info['L']), T=int(info['T']), val_rho_rfm=info['val_abs_rho'],
                         C=float(C), val_rho_logreg=rho_lr, cache_L=cL, cache_T=cT, cache_normalize=cN,
                         match_required=(cN == 0), match_cache=same,
                         cos_cache=abs(float(v @ V['v_per_block'][b])),
                         val_rho_cache=float(V['val_abs_rho'][b])))
hp = pd.DataFrame(rows); hp.to_csv(f'{PREFIX}_hp.csv', index=False)
print(f'step 0: {time.time() - t0:.0f}s'); display(hp)

bad = hp[hp.match_required & ~hp.match_cache]
assert bad.empty, f'C0 (step 0) FAILED: recomputed (L,T) differ from cache where hp_normalize=0:\n{bad}'
print('C0 step 0 passed: (L,T) reproduced in all', int(hp.match_required.sum()), 'cells with cached normalize=False')
print('primary block:', {c: HP[(c, B_PRIMARY)] for c in CONCEPTS})

## 1. Open the test split; draw subsamples (§3.2)

For each $(c,n)$, $R=200$ subsamples are drawn without replacement, $n$ positives and $n$ negatives from the train pool, seed `[SEED, concept index, n]`. Each subsample also carries the random pairing permutation used by PCA. The same subsamples are used for all four methods (paired design) and for all blocks.

**$n=n_{\max}$** (60 for `valence`, 180 otherwise) is the full pool: one evaluation. RFM, DM and LogReg are deterministic there. PCA depends on the pairing, which is fixed by the seed `[SEED, concept index, n_max]`, so the run is reproducible but the PCA value at $n_{\max}$ is one draw of the pairing.

In [ ]:
TE = {c: load_split(c, 'test') for c in CONCEPTS}
for c in CONCEPTS: print(f"{c:10s} test {TE[c][0].shape} pos {TE[c][1].sum()}")

def draw_subsamples(c):
    _, y, _ = TR[c]; P, Q = np.where(y == 1)[0], np.where(y == 0)[0]; ci = CONCEPTS.index(c)
    out = {}
    for n in N_GRID:
        rng = np.random.default_rng([SEED, ci, n])
        out[n] = [(rng.choice(P, n, replace=False), rng.choice(Q, n, replace=False), rng.permutation(n)) for _ in range(R)]
    nmax = min(len(P), len(Q))
    out[nmax] = [(P[:nmax], Q[:nmax], np.random.default_rng([SEED, ci, nmax]).permutation(nmax))]
    return out

SUBS = {c: draw_subsamples(c) for c in CONCEPTS}
NMAX = {c: max(SUBS[c]) for c in CONCEPTS}; print('n_max:', NMAX)

## 2. Methods on one subsample (§1.1, §3.3)

Preprocessing: centre on the subsample mean, `normalize=False`. Every direction is a unit vector oriented by the sign of the train Pearson correlation (`rfm.orient`).
- **RFM:** $M_0=I$, $T_{\max}=10$ calls to `_agop_step` with frozen $L$; the oriented top eigenvector of each $M_t$ is kept (§3.3.1). The primary direction uses the frozen $T$. Numerical rank of $M_T$: number of eigenvalues $>\lambda_{\max}\,d\,\varepsilon$.
- **DM:** $\hat\mu_+-\hat\mu_-$.
- **PCA:** *uncentred* (Appendix H of the paper): first right singular vector of the $n\times d$ matrix of paired differences (random pairing). Uncentred matters at $n=1$, where centring would give the zero matrix.
- **LogReg:** `liblinear`, $\ell_2$, frozen $C$, `random_state=42`.

**Residual** $\lVert P_{\mathcal D^\perp}v\rVert$: $\mathcal D$ = row space of the centred subsample, an orthonormal basis from the SVD, with rank tolerance $s_{\max}\max(N,d)\,\varepsilon$.

**Metrics (§3.4):** score $s(x)=\langle a_\ell(x),v\rangle$ on the full test split; AUROC (primary) and $|\operatorname{corr}(s,y)|$.

In [ ]:
EPS = np.finfo(np.float64).eps

def unit(v):
    return v / np.linalg.norm(v)

def span_basis(Zc):
    _, s, Vt = np.linalg.svd(Zc, full_matrices=False)
    return Vt[s > s.max() * max(Zc.shape) * EPS].T

def resid(v, Q):
    return float(np.linalg.norm(v - Q @ (Q.T @ v)))

def fit_directions(Zs, ys, L, T, C, perm):
    Zc, _ = _preprocess(Zs, True, False); yf = ys.astype(np.float64); d = Zc.shape[1]
    out = {'RFM_T': []}
    M = np.eye(d)
    for t in range(1, TMAX + 1):
        M, _, _ = _agop_step(Zc, yf, M, L, LAM)
        ev, U = np.linalg.eigh(M)
        out['RFM_T'].append(orient(U[:, -1], Zc, yf))
        if t == T:
            out['lam1'] = float(ev[-1]); out['rank_M'] = int((ev > ev[-1] * d * EPS).sum()) if ev[-1] > 0 else 0
    out['RFM'] = out['RFM_T'][T - 1]
    out['DM'] = orient(unit(Zc[ys == 1].mean(0) - Zc[ys == 0].mean(0)), Zc, yf)
    Dp = Zc[ys == 1][perm] - Zc[ys == 0]
    out['PCA'] = orient(np.linalg.svd(Dp, full_matrices=False)[2][0], Zc, yf)
    w = LogisticRegression(C=C, solver='liblinear', max_iter=2000, random_state=42).fit(Zc, ys).coef_[0]
    out['LogReg'] = orient(unit(w), Zc, yf)
    out['Q'] = span_basis(Zc)
    return out

def metrics(v, Zte, yte):
    s = Zte @ v
    return float(roc_auc_score(yte, s)), abs(float(np.corrcoef(s, yte)[0, 1]))

## 3. Main loop

One row per $(c,\ell,n,r,\text{method})$. RFM rows also carry `auc_T1..auc_T10` and `rho_T1..rho_T10` (§3.3.1), `resid_maxT`, the numerical rank of $M_T$ and $\lambda_1(M_T)$. **C0 is checked inside the loop**: a violation raises immediately. The partial table is saved after each block.

In [ ]:
rows = []; t0 = time.time()
for b in BLOCKS:
    for c in CONCEPTS:
        (Ztr, _, cat), (Zte, yte, _) = TR[c], TE[c]
        Ztr_b, Zte_b = Ztr[:, b], Zte[:, b]; hpc = HP[(c, b)]
        for n, draws in SUBS[c].items():
            for r, (pos, neg, perm) in enumerate(draws):
                ys = np.r_[np.ones(n, int), np.zeros(n, int)]
                out = fit_directions(Ztr_b[np.r_[pos, neg]], ys, hpc['L'], hpc['T'], hpc['C'], perm)
                Q = out['Q']
                base = dict(concept=c, block=b, n=n, r=r, dimD=Q.shape[1],
                            cov_pos=len(set(cat[pos])), cov_neg=len(set(cat[neg])))
                new = {}
                for m in METHODS:
                    v = out[m]; auc, rho = metrics(v, Zte_b, yte)
                    new[m] = dict(base, method=m, auc=auc, rho=rho, cos_dm=abs(float(v @ out['DM'])), resid=resid(v, Q))
                new['RFM'].update(L=hpc['L'], T=hpc['T'], rank_M=out['rank_M'], lam1=out['lam1'],
                                  resid_maxT=max(resid(vt, Q) for vt in out['RFM_T']))
                for t, vt in enumerate(out['RFM_T'], 1):
                    new['RFM'][f'auc_T{t}'], new['RFM'][f'rho_T{t}'] = metrics(vt, Zte_b, yte)
                new['LogReg']['C'] = hpc['C']
                # C0, fail-fast
                for m in METHODS:
                    assert new[m]['resid'] < TOL_RES, f'C0 FAILED (residual): {c} b{b} n={n} r={r} {m} resid={new[m]["resid"]:.2e}'
                if n == 1:
                    gap = 1 - new['RFM']['cos_dm']; dauc = abs(new['RFM']['auc'] - new['DM']['auc'])
                    assert gap < TOL_C0 and dauc < TOL_C0, f'C0 FAILED (n=1): {c} b{b} r={r} 1-|cos|={gap:.2e} |Δ|={dauc:.2e}'
                rows.extend(new.values())
        print(f'block {b} {c:10s} done  {time.time() - t0:.0f}s')
    DF = pd.DataFrame(rows); DF.to_csv(f'{PREFIX}_draws.csv', index=False)
print('draws:', DF.shape)

## 4. C0 summary

C0 was enforced in the loop; this cell reports the worst cases. Also reported, *not* part of C0: the $n=1$ coincidence of all four methods (Corollary 2), $\Delta(1,T)$ for every $T$, and the residual of RFM at every $T$ (Proposition 1 holds for every $t$).

In [ ]:
rf = DF[DF.method == 'RFM']; dm = DF[DF.method == 'DM'].set_index(['block', 'concept', 'n', 'r'])
rf1 = rf[rf.n == 1].set_index(['block', 'concept', 'n', 'r'])
c0 = {
    'n=1  max 1-|cos(RFM,DM)|': float((1 - rf1.cos_dm).max()),
    'n=1  max |AUROC RFM - DM|': float((rf1.auc - dm.loc[rf1.index].auc).abs().max()),
    **{f'max resid {m}': float(DF[DF.method == m].resid.max()) for m in METHODS},
    'max resid RFM over all T (info)': float(rf.resid_maxT.max()),
    **{f'n=1  max 1-|cos({m},DM)| (info)': float((1 - DF[(DF.method == m) & (DF.n == 1)].cos_dm).max()) for m in ['PCA', 'LogReg']},
    'n=1  max_T |Δ(1,T)| (info)': float(max((rf1[f'auc_T{t}'] - dm.loc[rf1.index].auc).abs().max() for t in range(1, TMAX + 1))),
}
for k, v in c0.items(): print(f'{k:40s} {v:.2e}')
print('C0 PASSED (step 0 + loop)')

## 5. Estimator, bootstrap CI, one-sided $p$, Holm (§3.5)

For each $(\ell,c,n)$ one bootstrap index matrix ($B=10{,}000$ resamples of the $R$ draws, seed `[SEED, 1, ℓ, concept index, n]`) is shared by all contrasts and metrics. Percentile 95% CI; $p_{c,n}=(1+\#\{b:\hat\Delta^*_b\le0\})/(1+B)$.

Holm is applied **within each block** to the $k=10$ cells (main concepts × $n\in\{2,4,8,16,32\}$), contrast RFM−DM, metric AUROC, $\alpha=0.025$. At $n_{\max}$ there is a single draw: $\hat\Delta$ is reported without a CI.

In [ ]:
CONTRASTS = [('RFM', 'DM'), ('RFM', 'LogReg'), ('RFM', 'PCA')]
TCOLS = [f'{k}_T{t}' for k in ('auc', 'rho') for t in range(1, TMAX + 1)]
W = {m: DF[DF.method == m].set_index(['block', 'concept', 'n', 'r']).sort_index() for m in METHODS}

def boot_idx(b, c, n):
    return np.random.default_rng([SEED, 1, b, CONCEPTS.index(c), n]).integers(0, R, size=(NBOOT, R))

def boot(d, idx):
    mb = d[idx].mean(1); lo, hi = np.percentile(mb, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi), float((1 + (mb <= 0).sum()) / (1 + NBOOT))

def holm(p, alpha=ALPHA):
    p = np.asarray(p); k = len(p); order = np.argsort(p, kind='stable')
    rank = np.empty(k, int); rank[order] = np.arange(1, k + 1); declared = np.zeros(k, bool)
    for j, i in enumerate(order, 1):
        if p[i] <= alpha / (k - j + 1): declared[i] = True
        else: break
    return rank, declared

srows, trows = [], []
for b in BLOCKS:
    for c in CONCEPTS:
        for n in N_GRID + [NMAX[c]]:
            key = (b, c, n); idx = boot_idx(*key) if n != NMAX[c] else None
            for a, z in CONTRASTS:
                for met in ('auc', 'rho'):
                    d = (W[a].loc[key][met] - W[z].loc[key][met]).to_numpy()
                    mean, lo, hi, p = boot(d, idx) if idx is not None else (float(d.mean()), np.nan, np.nan, np.nan)
                    srows.append(dict(block=b, concept=c, n=n, contrast=f'{a}-{z}', metric=met,
                                      delta=mean, ci_lo=lo, ci_hi=hi, p_one_sided=p))
            if idx is None: continue
            dmauc = W['DM'].loc[key]['auc'].to_numpy(); rfm = W['RFM'].loc[key]
            for t in range(1, TMAX + 1):
                mean, lo, hi, p = boot(rfm[f'auc_T{t}'].to_numpy() - dmauc, idx)
                trows.append(dict(block=b, concept=c, n=n, T=t, delta=mean, ci_lo=lo, ci_hi=hi, p_one_sided=p,
                                  is_T_frozen=(t == HP[(c, b)]['T'])))

SUM = pd.DataFrame(srows); SUM['holm_rank'] = np.nan; SUM['holm_threshold'] = np.nan; SUM['holm_declared'] = False
for b in BLOCKS:
    m = (SUM.block == b) & (SUM.contrast == 'RFM-DM') & (SUM.metric == 'auc') & SUM.concept.isin(MAIN) & SUM.n.isin(N_TEST)
    assert m.sum() == 10
    rank, dec = holm(SUM.loc[m, 'p_one_sided'].to_numpy())
    SUM.loc[m, 'holm_rank'] = rank; SUM.loc[m, 'holm_threshold'] = ALPHA / (10 - rank + 1); SUM.loc[m, 'holm_declared'] = dec
SUM.to_csv(f'{PREFIX}_summary.csv', index=False)

TS = pd.DataFrame(trows)
TS['is_T_star'] = False
for (b, c, n), g in TS.groupby(['block', 'concept', 'n']):
    TS.loc[g.delta.idxmax(), 'is_T_star'] = True           # first T among ties
TS.to_csv(f'{PREFIX}_T_sensitivity.csv', index=False)

prim = SUM[(SUM.block == B_PRIMARY) & (SUM.contrast == 'RFM-DM') & (SUM.metric == 'auc')]
display(prim[['concept', 'n', 'delta', 'ci_lo', 'ci_hi', 'p_one_sided', 'holm_rank', 'holm_threshold', 'holm_declared']].round(4))

## 6. Verdicts (§3.6)

H3b and the H-artigo family use $n\in\{2,4,8,16,32\}$: $n=1$ is identically $0$ (Corollary 2) and $n_{\max}$ has no CI. The table is printed for the primary block first; the other blocks follow with the same rules (secondary analysis, no selection). Saved to `IV-small-n_verdicts.csv`.

In [ ]:
def verdicts(b):
    S = SUM[(SUM.block == b) & (SUM.contrast == 'RFM-DM') & (SUM.metric == 'auc')]
    out = []
    h3a = S[S.concept.isin(MAIN) & S.n.isin([2, 4])]
    hit = h3a[h3a.ci_lo > 0]
    out.append(dict(block=b, hypothesis='H3a', verdict='REFUTED' if len(hit) else 'not refuted',
                    evidence='; '.join(f'{r.concept} n={r.n} lo={r.ci_lo:.4f}' for r in hit.itertuples()) or 'no lower bound > 0'))
    dec = S[S.holm_declared]
    out.append(dict(block=b, hypothesis='H-artigo', verdict='SUPPORTED' if len(dec) else 'REFUTED',
                    evidence='; '.join(f'{r.concept} n={r.n} p={r.p_one_sided:.4g}' for r in dec.itertuples()) or
                             f'min p = {S[S.concept.isin(MAIN) & S.n.isin(N_TEST)].p_one_sided.min():.4g} > {ALPHA / 10:.4g}'))
    ev = []
    for c in MAIN:
        s = S[(S.concept == c) & S.n.isin(N_TEST)].sort_values('n')
        ev += [f'{c}: n={i.n}<{j.n}' for i in s.itertuples() for j in s.itertuples() if i.n < j.n and i.ci_hi < 0 and j.ci_lo > 0]
    out.append(dict(block=b, hypothesis='H3b', verdict='SUPPORTED' if ev else 'not supported', evidence='; '.join(ev) or '—'))
    ts = TS[(TS.block == b) & TS.is_T_star & TS.concept.isin(MAIN) & TS.n.isin([2, 4])]
    ok = ts[ts.ci_hi < 0]
    out.append(dict(block=b, hypothesis='H3a reinforcement (oracle T)', verdict='YES' if len(ok) else 'no',
                    evidence='; '.join(f'{r.concept} n={r.n} T*={r.T} Δmax={r.delta:.4f} hi={r.ci_hi:.4f}' for r in ts.itertuples())))
    return out

VER = pd.DataFrame([v for b in BLOCKS for v in verdicts(b)])
VER.to_csv(f'{PREFIX}_verdicts.csv', index=False)
pd.set_option('display.max_colwidth', 200); display(VER)

## 7. Figures (primary block)

`IV-small-n_curves.png`: top, mean test AUROC of the four methods vs $n$ (95% bootstrap CI of the mean; $n_{\max}$ as an isolated marker); bottom, $\hat\Delta_c(n)$ RFM−DM with its CI band.
`IV-small-n_T_curves.png`: $\hat\Delta_c(n,T)$ vs $T$, one curve per $n$; dashed line at the frozen $T$, star at $T^\star_{c,n}$.

In [ ]:
b = B_PRIMARY; colors = dict(RFM='tab:red', DM='tab:blue', PCA='tab:green', LogReg='tab:purple')
fig, axs = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for j, c in enumerate(CONCEPTS):
    ax = axs[0, j]
    for m in METHODS:
        mu, lo, hi = [], [], []
        for n in N_GRID:
            a = W[m].loc[(b, c, n)]['auc'].to_numpy(); mb = a[boot_idx(b, c, n)].mean(1)
            mu.append(a.mean()); lo.append(np.percentile(mb, 2.5)); hi.append(np.percentile(mb, 97.5))
        ax.plot(N_GRID, mu, 'o-', color=colors[m], label=m); ax.fill_between(N_GRID, lo, hi, color=colors[m], alpha=.15)
        ax.plot([NMAX[c]], [W[m].loc[(b, c, NMAX[c])]['auc'].iloc[0]], 's', color=colors[m])
    ax.set_title(f'{c}  block {b}'); ax.set_ylabel('test AUROC'); ax.set_xscale('log', base=2)
    ax = axs[1, j]; s = SUM[(SUM.block == b) & (SUM.concept == c) & (SUM.contrast == 'RFM-DM') & (SUM.metric == 'auc')].sort_values('n')
    g = s[s.n.isin(N_GRID)]
    ax.plot(g.n, g.delta, 'o-', color='k'); ax.fill_between(g.n, g.ci_lo, g.ci_hi, color='k', alpha=.15)
    ax.plot([NMAX[c]], s[s.n == NMAX[c]].delta, 's', color='k'); ax.axhline(0, ls=':', color='gray')
    ax.set_xlabel('n per class'); ax.set_ylabel(r'$\hat\Delta$ AUROC (RFM − DM)')
axs[0, 0].legend(); plt.tight_layout(); plt.savefig(f'{PREFIX}_curves.png', dpi=120); plt.show()

fig, axs = plt.subplots(1, 3, figsize=(16, 4.5), sharey=False)
cmap = plt.get_cmap('viridis')
for j, c in enumerate(CONCEPTS):
    ax = axs[j]
    for i, n in enumerate(N_GRID):
        g = TS[(TS.block == b) & (TS.concept == c) & (TS.n == n)].sort_values('T')
        ax.plot(g['T'], g.delta, 'o-', ms=3, color=cmap(i / (len(N_GRID) - 1)), label=f'n={n}')
        st = g[g.is_T_star]; ax.plot(st['T'], st.delta, '*', ms=12, color=cmap(i / (len(N_GRID) - 1)))
    ax.axvline(HP[(c, b)]['T'], ls='--', color='gray'); ax.axhline(0, ls=':', color='gray')
    ax.set_title(f'{c}  block {b}  (frozen T={HP[(c, b)]["T"]})'); ax.set_xlabel('T'); ax.set_ylabel(r'$\hat\Delta(n,T)$ AUROC')
axs[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{PREFIX}_T_curves.png', dpi=120); plt.show()

## 8. Outputs
`IV-small-n_hp.csv`, `IV-small-n_draws.csv`, `IV-small-n_summary.csv`, `IV-small-n_T_sensitivity.csv`, `IV-small-n_verdicts.csv`, `IV-small-n_curves.png`, `IV-small-n_T_curves.png`. The report `IV-small-n.md` is written by hand from these files.